In [ ]:
import os
import itertools
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LogNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from scipy import stats

# -------------------- User paths --------------------
nc_path   = r"C:\Drought\Regridding and data clipping\VDI_v02_ESI\VDI_with_ESI_monthly_India_0p05deg.nc"
shp_path  = r"C:\Drought\India Shapefile\indiashapefile\India_with_jk.shp"
out_dir   = r"C:\Drought\Figure"

os.makedirs(out_dir, exist_ok=True)

# -------------------- Load data --------------------
print("Loading NetCDF...")
ds = xr.open_dataset(nc_path)
# variables present (screenshot-based)
# TVDI_dryz, VOD_dryz, NDVI_dryz, SIF_dryz, SMsurf_dryz, SMroot_dryz, ESI_dryz, VDI, VDI_uncert
all_vars = [v for v in ds.data_vars if v != "VDI_uncert"]
var_names = all_vars  # keep order as in file

# Basic dims sanity
expected_dims = set(["time","lat","lon"])
for v in var_names:
    if set(ds[v].dims) != expected_dims:
        raise ValueError(f"Variable {v} dims are {ds[v].dims}, expected (time, lat, lon) in any order.")

# Read shapefile for overlay
print("Loading India shapefile...")
gdf_india = gpd.read_file(shp_path)
# Transform to PlateCarree (lon/lat) coordinate reference
try:
    gdf_india = gdf_india.to_crs(epsg=4326)
except Exception:
    pass

# -------------------- Helper functions --------------------
def cosine_lat_weights(lat):
    """Return 1D cosine-lat weights normalized to mean 1.0 (so spatial mean is comparable)."""
    w = np.cos(np.deg2rad(lat))
    return w / w.mean()

def plot_spatial(ax, da, title, vmin=-1, vmax=1, cmap="RdBu_r", add_coast=True, shp=None):
    """Generic spatial map plotter with shapefile overlay."""
    proj = ccrs.PlateCarree()
    ax.set_extent([float(ds.lon.min()), float(ds.lon.max()), float(ds.lat.min()), float(ds.lat.max())], crs=proj)
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    im = da.plot.pcolormesh(ax=ax, transform=proj, cmap=cmap, norm=norm, add_colorbar=False)
    if shp is not None:
        shp.boundary.plot(ax=ax, transform=proj, linewidth=0.6, edgecolor="black", zorder=3)
    if add_coast:
        ax.coastlines(resolution="10m", linewidth=0.5)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    return im

def pearson_r_over_time(a, b):
    """Compute Pearson r along time dimension for 3D arrays time x lat x lon."""
    # reshape to time x space
    T, Y, X = a.shape
    A = a.reshape(T, -1)
    B = b.reshape(T, -1)
    # mask where all-NaN
    mask = np.any(np.isfinite(A), axis=0) & np.any(np.isfinite(B), axis=0)
    r = np.full(A.shape[1], np.nan, dtype=float)
    for i in np.where(mask)[0]:
        aa = A[:, i]; bb = B[:, i]
        m = np.isfinite(aa) & np.isfinite(bb)
        if m.sum() >= 10:
            r[i] = stats.pearsonr(aa[m], bb[m]).statistic
    return r.reshape(Y, X)

def lag_array(x, lag):
    """Lag x by N (positive lag means x leads). For correlation of X_lag with Y at same timestamps."""
    if lag <= 0:
        return x
    y = np.full_like(x, np.nan)
    y[lag:] = x[:-lag]
    return y

def theil_sen_slope(y, x):
    """Fallback to simple linear slope if few points; returns slope per unit x."""
    if np.sum(np.isfinite(y)) < 6:
        return np.nan
    slope, intercept, r, p, se = stats.linregress(x[np.isfinite(y)], y[np.isfinite(y)])
    return slope

def monthly_to_season(da, season):
    """Seasonal mean with DJF handling; returns DataArray with 'year' dimension."""
    if season == "Annual":
        return da.groupby("time.year").mean("time")
    if season == "DJF":
        # shift Dec to next year
        m = da["time.month"]
        shifted_year = xr.where(m==12, da["time.year"]+1, da["time.year"])
        return da.groupby(shifted_year.rename("year")).apply(lambda x: x.sel(time=x["time.month"].isin([12,1,2])).mean("time"))
    if season == "MAM":
        return da.sel(time=da["time.month"].isin([3,4,5])).groupby("time.year").mean("time")
    if season == "JJAS":
        return da.sel(time=da["time.month"].isin([6,7,8,9])).groupby("time.year").mean("time")
    if season == "ON":
        return da.sel(time=da["time.month"].isin([10,11])).groupby("time.year").mean("time")
    raise ValueError("Unknown season")

def annotate_cbar(fig, im, label, shrink=0.6):
    cbar = fig.colorbar(im, ax=fig.axes, orientation="horizontal", fraction=0.05, pad=0.04, shrink=shrink)
    cbar.set_label(label, fontsize=9)
    cbar.ax.tick_params(labelsize=8)

# -------------------- 1) Spatial correlations for all pairs --------------------
print("Computing spatial correlations for all pairs...")
pairs = list(itertools.combinations(var_names, 2))
n = len(pairs)

# Decide figure grid (rows x cols) close to square; literature-style small multiples
cols = 7  # 28 panels → 4 rows × 7 cols typical compact layout
rows = int(np.ceil(n/cols))

proj = ccrs.PlateCarree()
fig = plt.figure(figsize=(cols*2.2, rows*2.2))
axes = [fig.add_subplot(rows, cols, i+1, projection=proj) for i in range(rows*cols)]

vmin, vmax = -1.0, 1.0
last_im = None
for ax, (a, b) in zip(axes, pairs):
    A = ds[a].transpose("time","lat","lon").values
    B = ds[b].transpose("time","lat","lon").values
    rmap = pearson_r_over_time(A, B)
    da = xr.DataArray(rmap, coords={"lat": ds.lat, "lon": ds.lon}, dims=("lat","lon"))
    title = f"{a} vs {b}"
    last_im = plot_spatial(ax, da, title, vmin=vmin, vmax=vmax, shp=gdf_india)
# empty extra axes
for ax in axes[len(pairs):]:
    ax.axis("off")

fig.suptitle("Spatial Pearson r (per-pixel, over time) — All pairs (excl. VDI_uncert)", fontsize=12, y=0.92)
if last_im is not None:
    cax = fig.add_axes([0.25, 0.06, 0.5, 0.02])
    cbar = fig.colorbar(last_im, cax=cax, orientation="horizontal")
    cbar.set_label("Pearson r")
    cbar.ax.tick_params(labelsize=8)

out1 = os.path.join(out_dir, "Fig1_spatial_corr_all_pairs.png")
plt.tight_layout(rect=[0,0.08,1,0.9])
plt.savefig(out1, dpi=300)
plt.close(fig)
print(f"Saved: {out1}")

# -------------------- 2) Lag correlations for SMsurf/SMroot vs others --------------------
print("Computing lag correlations (1–3 months) for SMsurf_dryz and SMroot_dryz...")
ref_vars = ["SMsurf_dryz", "SMroot_dryz"]
targets = [v for v in var_names if v not in ref_vars]

for ref in ref_vars:
    A = ds[ref].transpose("time","lat","lon").values
    for lag in [1,2,3]:
        print(f"  {ref} at lag {lag}")
        cols = 7
        rows = int(np.ceil(len(targets)/cols))
        fig = plt.figure(figsize=(cols*2.2, rows*2.2))
        axes = [fig.add_subplot(rows, cols, i+1, projection=proj) for i in range(rows*cols)]
        last_im = None
        for ax, tgt in zip(axes, targets):
            B = ds[tgt].transpose("time","lat","lon").values
            A_lag = lag_array(A, lag)  # ref leads by 'lag' months
            rmap = pearson_r_over_time(A_lag, B)
            da = xr.DataArray(rmap, coords={"lat": ds.lat, "lon": ds.lon}, dims=("lat","lon"))
            title = f"{ref}(t-{lag}) vs {tgt}"
            last_im = plot_spatial(ax, da, title, vmin=vmin, vmax=vmax, shp=gdf_india)
        for ax in axes[len(targets):]:
            ax.axis("off")
        fig.suptitle(f"Spatial Pearson r — {ref} lead by {lag} mo (vs others)", fontsize=12, y=0.92)
        if last_im is not None:
            cax = fig.add_axes([0.25, 0.06, 0.5, 0.02])
            cbar = fig.colorbar(last_im, cax=cax, orientation="horizontal")
            cbar.set_label("Pearson r")
        outp = os.path.join(out_dir, f"Fig2_lagcorr_{ref}_lag{lag}.png")
        plt.tight_layout(rect=[0,0.08,1,0.9])
        plt.savefig(outp, dpi=300)
        plt.close(fig)
        print(f"Saved: {outp}")

# -------------------- 3) Scatter plots (hexbin density) for all pairs --------------------
print("Generating density scatter (hexbin) plots for all pairs...")
# Sample to avoid gigantic memory: pick 5000 random pixels and use full time
rng = np.random.default_rng(42)
ny, nx = len(ds.lat), len(ds.lon)
flat_idx = np.arange(ny*nx)
sample_n = min(5000, flat_idx.size)
sample_pix = rng.choice(flat_idx, size=sample_n, replace=False)

def flatten_space_time(da):
    arr = da.values  # (time, lat, lon)
    T, Y, X = arr.shape
    return arr.reshape(T, Y*X)[:, sample_pix].ravel()

pairs = list(itertools.combinations(var_names, 2))
cols = 6
rows = int(np.ceil(len(pairs)/cols))
fig = plt.figure(figsize=(cols*3.0, rows*2.6))
for i, (a,b) in enumerate(pairs, start=1):
    ax = fig.add_subplot(rows, cols, i)
    x = flatten_space_time(ds[a].transpose("time","lat","lon"))
    y = flatten_space_time(ds[b].transpose("time","lat","lon"))
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 50:
        continue
    hb = ax.hexbin(x[m], y[m], gridsize=40, bins='log')
    ax.set_title(f"{a} vs {b}", fontsize=9)
    ax.set_xlabel(a, fontsize=8)
    ax.set_ylabel(b, fontsize=8)
    ax.tick_params(labelsize=8)
cbar = fig.colorbar(hb, ax=fig.axes, orientation="horizontal", fraction=0.05, pad=0.04)
cbar.set_label("log10(count)", fontsize=9)
out3 = os.path.join(out_dir, "Fig3_hexbin_scatter_all_pairs.png")
plt.tight_layout()
plt.savefig(out3, dpi=300)
plt.close(fig)
print(f"Saved: {out3}")

# -------------------- 4) India-mean time series (bars with gradient) --------------------
print("Computing area-weighted India-mean time series...")
w = cosine_lat_weights(ds.lat.values)
# expand to 2D weights lat x lon
W = xr.DataArray(w, coords={"lat": ds.lat}, dims="lat")
def area_mean(da):
    return (da * W).sum(dim=("lat"))/W.sum(dim="lat")

series = {}
for v in var_names:
    series[v] = area_mean(ds[v]).mean(dim="lon")
df = xr.Dataset(series).to_dataframe()
df.index = pd.to_datetime(df.index)

# Plot bars per variable in a grid
cols = 2
rows = int(np.ceil(len(var_names)/cols))
fig = plt.figure(figsize=(cols*8, rows*2.6))
for i, v in enumerate(var_names, start=1):
    ax = fig.add_subplot(rows, cols, i)
    vals = df[v].copy()
    # Colour by value using colormap; draw bars
    cmap = plt.get_cmap("RdBu_r")
    norm = TwoSlopeNorm(vmin=np.nanpercentile(vals, 5), vcenter=0, vmax=np.nanpercentile(vals,95))
    colors = cmap(norm(vals.values))
    ax.bar(df.index, vals.values, width=25, align="center", color=colors, edgecolor="none")
    ax.set_title(f"{v} — India mean", fontsize=10)
    ax.tick_params(labelsize=8)
    ax.set_xlim(df.index.min()-pd.Timedelta(days=15), df.index.max()+pd.Timedelta(days=15))
fig.autofmt_xdate()
out4 = os.path.join(out_dir, "Fig4_timeseries_bars_india_mean.png")
plt.tight_layout()
plt.savefig(out4, dpi=300)
plt.close(fig)
print(f"Saved: {out4}")

# -------------------- 5) Trends (annual & seasonal) --------------------
print("Estimating spatial linear trends per decade...")
seasons = ["Annual", "DJF", "MAM", "JJAS", "ON"]
for v in var_names:
    print(f"  Trends for {v}")
    for s in seasons:
        da_seas = monthly_to_season(ds[v], s)  # dims: year, lat, lon
        years = da_seas["year"].values.astype(float)
        # compute slope per pixel
        arr = da_seas.transpose("year","lat","lon").values  # Y x Ny x Nx
        Y, Ny, Nx = arr.shape
        slopes = np.full((Ny, Nx), np.nan, dtype=float)
        for j in range(Ny):
            for i in range(Nx):
                y = arr[:, j, i]
                if np.isfinite(y).sum() >= max(8, int(0.6*Y)):
                    slopes[j,i] = theil_sen_slope(y, years) * 10.0  # per decade
        da_slope = xr.DataArray(slopes, coords={"lat": ds.lat, "lon": ds.lon}, dims=("lat","lon"))
        fig = plt.figure(figsize=(6,4))
        ax = fig.add_subplot(1,1,1, projection=proj)
        im = plot_spatial(ax, da_slope, f"{v} Trend ({s}) [unit/decade]", vmin=np.nanpercentile(slopes,5), vmax=np.nanpercentile(slopes,95), cmap="RdBu_r", shp=gdf_india)
        cbar = fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.046, pad=0.06)
        cbar.set_label("Slope per decade")
        outp = os.path.join(out_dir, f"Fig5_trend_{v}_{s}.png")
        plt.tight_layout()
        plt.savefig(outp, dpi=300)
        plt.close(fig)
        print(f"Saved: {outp}")

# -------------------- 6) Annual cycle (India-mean) with 5–95% band --------------------
print("Computing annual cycle with 5–95% ranges...")
fig = plt.figure(figsize=(10, 2.2*len(var_names)))
for i, v in enumerate(var_names, start=1):
    ax = fig.add_subplot(len(var_names), 1, i)
    da = ds[v]
    mgrp = da.groupby("time.month")
    # area-weighted mean for each month
    clim = mgrp.map(area_mean).mean("lon")
    # compute 5–95% across years for each calendar month
    by_month_year = da.groupby("time.month").map(area_mean).groupby("time.year").mean("lon")  # year x month
    lo = by_month_year.quantile(0.05, dim="year")
    hi = by_month_year.quantile(0.95, dim="year")
    months = np.arange(1,13)
    ax.fill_between(months, lo, hi, alpha=0.25, linewidth=0)
    ax.plot(months, clim, lw=1.5)
    ax.set_xticks(months); ax.set_xlim(1,12)
    ax.set_title(f"{v} Annual Cycle (India mean) — shaded 5–95%", fontsize=9)
    ax.grid(True, alpha=0.3, linestyle=":")
out6 = os.path.join(out_dir, "Fig6_annual_cycle_with_spread.png")
plt.tight_layout()
plt.savefig(out6, dpi=300)
plt.close(fig)
print(f"Saved: {out6}")

# -------------------- 7) Violin/Box stats by month over India --------------------
print("Building violin/box plots by month over India...")
# Build dataframe: columns [variable, month, value] aggregated over space (India mean) each month
records = []
for v in var_names:
    vals = area_mean(ds[v]).mean("lon")
    dfv = vals.to_pandas()
    for ts, val in dfv.items():
        records.append((v, ts.month, float(val)))
df_all = pd.DataFrame(records, columns=["variable","month","value"])

# Violin plots per variable with overlaid box
cols = 2
rows = int(np.ceil(len(var_names)/cols))
fig = plt.figure(figsize=(cols*7, rows*3))
for i, v in enumerate(var_names, start=1):
    ax = fig.add_subplot(rows, cols, i)
    sub = df_all[df_all["variable"]==v]
    data = [sub[sub["month"]==m]["value"].values for m in range(1,13)]
    parts = ax.violinplot(data, positions=np.arange(1,13), showmeans=False, showextrema=False, widths=0.8)
    # box-style stats
    q05 = [np.nanpercentile(d,5) if len(d)>0 else np.nan for d in data]
    q50 = [np.nanpercentile(d,50) if len(d)>0 else np.nan for d in data]
    q95 = [np.nanpercentile(d,95) if len(d)>0 else np.nan for d in data]
    ax.plot(np.arange(1,13), q50, lw=1.2)
    ax.fill_between(np.arange(1,13), q05, q95, alpha=0.25, linewidth=0)
    ax.set_title(f"{v} — Monthly distribution (India mean)", fontsize=9)
    ax.set_xlim(0.5, 12.5)
    ax.set_xticks(np.arange(1,13))
    ax.grid(True, alpha=0.2, linestyle=":")
out7 = os.path.join(out_dir, "Fig7_violin_box_monthly_india_mean.png")
plt.tight_layout()
plt.savefig(out7, dpi=300)
plt.close(fig)
print(f"Saved: {out7}")

print("All tasks completed. Figures saved to:", out_dir)


In [15]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import os

# --- Robust symmetric limits around zero to keep TwoSlopeNorm happy ---
def diverging_limits_from_percentiles(arr, lo_q=5, hi_q=95, eps=1e-8):
    """
    Return symmetric (-m, +m) around 0 using percentile envelope.
    Ensures vmin < 0 < vmax even if slopes are all positive or all negative.
    """
    lo = np.nanpercentile(arr, lo_q)
    hi = np.nanpercentile(arr, hi_q)
    m = max(abs(lo), abs(hi))
    if not np.isfinite(m) or m == 0:
        # fall back to absolute max; if still zero, use tiny epsilon
        m = max(np.nanmax(np.abs(arr)), eps)
    return -m, m

# -------------------------------- 5) Trends (resume) --------------------------------
print("Resuming: spatial linear trends per decade (robust color limits, skip existing)…")
seasons = ["Annual", "DJF", "MAM", "JJAS", "ON"]

for v in var_names:
    print(f"  Trends for {v}")
    for s in seasons:
        outp = os.path.join(out_dir, f"Fig5_trend_{v}_{s}.png")
        if os.path.exists(outp):
            print("    Exists, skip ->", outp)
            continue

        da_seas = monthly_to_season(ds[v], s)  # (year, lat, lon)
        years = da_seas["year"].values.astype(float)

        arr = da_seas.transpose("year", "lat", "lon").values  # Y x Ny x Nx
        Y, Ny, Nx = arr.shape
        slopes = np.full((Ny, Nx), np.nan, dtype=float)

        # per-pixel slope (linregress) → per decade
        for j in range(Ny):
            for i in range(Nx):
                y = arr[:, j, i]
                if np.isfinite(y).sum() >= max(8, int(0.6 * Y)):
                    slopes[j, i] = theil_sen_slope(y, years) * 10.0

        vmin, vmax = diverging_limits_from_percentiles(slopes, 5, 95)
        da_slope = xr.DataArray(
            slopes, coords={"lat": ds.lat, "lon": ds.lon}, dims=("lat", "lon")
        )

        fig = plt.figure(figsize=(6, 4))
        ax = fig.add_subplot(1, 1, 1, projection=proj)
        im = plot_spatial(
            ax,
            da_slope,
            f"{v} Trend ({s}) [unit/decade]",
            vmin=vmin,
            vmax=vmax,
            cmap="RdBu_r",
            shp=gdf_india,
        )
        cbar = fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.046, pad=0.06)
        cbar.set_label("Slope per decade")
        plt.tight_layout()
        plt.savefig(outp, dpi=300)
        plt.close(fig)
        print("    Saved:", outp)

# ----------------------- 6) Annual cycle (India-mean) with 5–95% band -----------------------
print("Now: annual cycle with 5–95% shading (India mean)…")
fig = plt.figure(figsize=(10, 2.2 * len(var_names)))
for i, v in enumerate(var_names, start=1):
    ax = fig.add_subplot(len(var_names), 1, i)
    da = ds[v]

    # India-mean time series first
    ts = area_mean(da).mean("lon").to_pandas()

    # monthly climatology and spread across years
    clim = ts.groupby(ts.index.month).mean()
    spread = ts.groupby([ts.index.year, ts.index.month]).mean().unstack(0)
    lo = spread.quantile(0.05, axis=1)
    hi = spread.quantile(0.95, axis=1)

    months = np.arange(1, 13, dtype=int)
    ax.fill_between(months, lo.values, hi.values, alpha=0.25, linewidth=0)
    ax.plot(months, clim.values, lw=1.5)
    ax.set_xlim(1, 12)
    ax.set_xticks(months)
    ax.set_title(f"{v} — Annual cycle (India mean), shaded 5–95%")
    ax.grid(True, alpha=0.3, linestyle=":")

out6 = os.path.join(out_dir, "Fig6_annual_cycle_with_spread.png")
plt.tight_layout()
plt.savefig(out6, dpi=300)
plt.close(fig)
print("Saved:", out6)

# ---------------- Violin/Box (p05–median–p95) by month (India mean) ----------------
print("Finally: violin + box (p05/50/95) by month (India mean)…")

# Build dataframe [variable, month, value] from India-mean monthly series
import pandas as pd
records = []
for v in var_names:
    ts = area_mean(ds[v]).mean("lon").to_pandas()
    for t, val in ts.items():
        records.append((v, t.month, float(val)))
df_all = pd.DataFrame(records, columns=["variable", "month", "value"])

cols = 2
rows = int(np.ceil(len(var_names) / cols))
fig = plt.figure(figsize=(cols * 7, rows * 3))
for i, v in enumerate(var_names, start=1):
    ax = fig.add_subplot(rows, cols, i)
    sub = df_all[df_all["variable"] == v]
    data = [sub[sub["month"] == m]["value"].values for m in range(1, 13)]

    parts = ax.violinplot(
        data, positions=np.arange(1, 13), showmeans=False, showextrema=False, widths=0.85
    )

    q05 = [np.nanpercentile(d, 5) if len(d) else np.nan for d in data]
    q50 = [np.nanpercentile(d, 50) if len(d) else np.nan for d in data]
    q95 = [np.nanpercentile(d, 95) if len(d) else np.nan for d in data]

    ax.plot(np.arange(1, 13), q50, lw=1.2)
    ax.fill_between(np.arange(1, 13), q05, q95, alpha=0.25, linewidth=0)
    ax.set_xlim(0.5, 12.5)
    ax.set_xticks(np.arange(1, 13))
    ax.set_title(f"{v} — Monthly distribution (India mean)")
    ax.grid(True, alpha=0.2, linestyle=":")

out7 = os.path.join(out_dir, "Fig7_violin_box_monthly_india_mean.png")
plt.tight_layout()
plt.savefig(out7, dpi=300)
plt.close(fig)
print("Saved:", out7)

print("\nDone. Trends (remaining seasons/vars) + Fig6 + Fig7 have been produced where missing.")


Resuming: spatial linear trends per decade (robust color limits, skip existing)…
  Trends for TVDI_dryz
    Exists, skip -> C:\Drought\Figure\Fig5_trend_TVDI_dryz_Annual.png
    Exists, skip -> C:\Drought\Figure\Fig5_trend_TVDI_dryz_DJF.png
    Exists, skip -> C:\Drought\Figure\Fig5_trend_TVDI_dryz_MAM.png
    Exists, skip -> C:\Drought\Figure\Fig5_trend_TVDI_dryz_JJAS.png
    Exists, skip -> C:\Drought\Figure\Fig5_trend_TVDI_dryz_ON.png
  Trends for VOD_dryz
    Exists, skip -> C:\Drought\Figure\Fig5_trend_VOD_dryz_Annual.png
    Exists, skip -> C:\Drought\Figure\Fig5_trend_VOD_dryz_DJF.png
    Exists, skip -> C:\Drought\Figure\Fig5_trend_VOD_dryz_MAM.png
    Exists, skip -> C:\Drought\Figure\Fig5_trend_VOD_dryz_JJAS.png
    Saved: C:\Drought\Figure\Fig5_trend_VOD_dryz_ON.png
  Trends for NDVI_dryz
    Saved: C:\Drought\Figure\Fig5_trend_NDVI_dryz_Annual.png
    Saved: C:\Drought\Figure\Fig5_trend_NDVI_dryz_DJF.png
    Saved: C:\Drought\Figure\Fig5_trend_NDVI_dryz_MAM.png
    Saved: C